In [10]:
# 从上周开发的 mysql_helper.py 文件中导入 MySqlHelper 类
# MySqlHelper 已经封装好了 MySQL 的连接、查询、插入、更新、删除等操作
from mysql_helper import MySqlHelper

# requests 用于向网页发送请求，获取网页 HTML 内容
import requests

# BeautifulSoup 用于解析 HTML，从网页中提取需要的数据
from bs4 import BeautifulSoup

# datetime 用于记录当前爬取时间
from datetime import datetime

In [11]:
# 使用 with 语法创建 MySqlHelper 对象
# 好处是：代码执行结束后会自动关闭数据库连接
with MySqlHelper() as db:

    # 查询当前连接的是哪个数据库
    # 如果 .env 中 DB_NAME=school_db，这里应该输出 school_db
    result = db.query_one("SELECT DATABASE() AS current_database;")

    # 打印当前数据库名称，确认 Python 已经成功连接 MySQL
    print("当前连接的数据库：", result["current_database"])

当前连接的数据库： school_db


In [12]:
def crawl_baidu_hot_search_top10():
    """
    爬取百度热搜榜前 10 条数据。
    返回值：列表 list
    列表中的每个元素是一个字典 dict，包含排名、关键词、热度、链接、爬取时间。
    """

    # 百度热搜实时榜页面地址
    url = "https://top.baidu.com/board?tab=realtime"

    # 设置请求头，模拟浏览器访问
    # 有些网站会拒绝没有 User-Agent 的请求
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    # 向百度热搜页面发送 GET 请求
    # timeout=10 表示最多等待 10 秒，避免程序一直卡住
    response = requests.get(url, headers=headers, timeout=10)

    # 如果请求失败，例如 403、404、500，这一行会抛出异常
    response.raise_for_status()

    # 设置编码，避免中文乱码
    response.encoding = "utf-8"

    # 使用 BeautifulSoup 解析 HTML 页面
    soup = BeautifulSoup(response.text, "html.parser")

    # 创建空列表，用来保存爬取到的热搜数据
    hot_list = []

    # 找到页面中每一条热搜对应的 HTML 区块
    # 注意：如果百度页面结构变化，这个选择器可能需要调整
    items = soup.select(".category-wrap_iQLoo")

    # 遍历前 10 条热搜
    # enumerate(..., start=1) 会自动生成排名 1、2、3...
    # 这样比从网页里提取排名更稳定，避免出现 int('') 的错误
    for rank_no, item in enumerate(items[:10], start=1):

        # 提取热搜关键词所在的标签
        title_tag = item.select_one(".c-single-text-ellipsis")

        # 提取热度值所在的标签
        hot_tag = item.select_one(".hot-index_1Bl1a")

        # 提取链接所在的 a 标签
        link_tag = item.select_one("a")

        # 如果没有找到标题，说明这一条数据不完整，跳过
        if not title_tag:
            continue

        # 获取热搜关键词文本
        keyword = title_tag.get_text(strip=True)

        # 去掉标题中可能混入的“热”“新”等标记
        keyword = keyword.replace("热", "").replace("新", "").strip()

        # 获取热度值，如果没找到就用 None
        hot_value = hot_tag.get_text(strip=True) if hot_tag else None

        # 获取热搜链接，如果没找到就用 None
        link = link_tag.get("href") if link_tag else None

        # 把一条热搜数据整理成字典，方便后面存入数据库
        hot_list.append({
            "rank_no": rank_no,
            "keyword": keyword,
            "hot_value": hot_value,
            "url": link,
            "crawl_time": datetime.now()
        })

    # 返回最终爬取结果
    return hot_list


# 调用函数，开始爬取百度热搜 top10
hot_search_data = crawl_baidu_hot_search_top10()

# 打印爬取结果，先检查是否成功爬到数据
for item in hot_search_data:
    print(item)

{'rank_no': 1, 'keyword': '习近平同金正恩举行会谈', 'hot_value': '7904592', 'url': 'https://www.baidu.com/s?wd=%E4%B9%A0%E8%BF%91%E5%B9%B3%E5%90%8C%E9%87%91%E6%AD%A3%E6%81%A9%E4%B8%BE%E8%A1%8C%E4%BC%9A%E8%B0%88&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34, 426343)}
{'rank_no': 2, 'keyword': '习近平和彭丽媛观看朝鲜文艺演出', 'hot_value': '7808599', 'url': 'https://www.baidu.com/s?wd=%E4%B9%A0%E8%BF%91%E5%B9%B3%E5%92%8C%E5%BD%AD%E4%B8%BD%E5%AA%9B%E8%A7%82%E7%9C%8B%E6%9C%9D%E9%B2%9C%E6%96%87%E8%89%BA%E6%BC%94%E5%87%BA&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34, 426465)}
{'rank_no': 3, 'keyword': '“李华 我再也不用见到你了”', 'hot_value': '7712263', 'url': 'https://www.baidu.com/s?wd=%E2%80%9C%E6%9D%8E%E5%8D%8E+%E6%88%91%E5%86%8D%E4%B9%9F%E4%B8%8D%E7%94%A8%E8%A7%81%E5%88%B0%E4%BD%A0%E4%BA%86%E2%80%9D&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34, 426575)}
{'rank_no': 4, 'keyword': '中朝传统友谊崭篇章如何续写', 'hot_value

In [13]:
# 定义插入 SQL
# baidu_hot_search 是你已经在 school_db 中创建好的表
# rank_no、keyword、hot_value、url、crawl_time 是表中的字段
# %s 是占位符，真实数据会通过 params_list 传入
insert_sql = """
INSERT INTO baidu_hot_search (rank_no, keyword, hot_value, url, crawl_time)
VALUES (%s, %s, %s, %s, %s)
"""


# 将 hot_search_data 中的字典数据转换成元组列表
# execute_many 需要的数据格式类似：
# [
#     (1, "热搜关键词1", "123456", "https://xxx", 时间),
#     (2, "热搜关键词2", "234567", "https://xxx", 时间)
# ]
params_list = [
    (
        item["rank_no"],      # 热搜排名
        item["keyword"],      # 热搜关键词
        item["hot_value"],    # 热度值
        item["url"],          # 热搜链接
        item["crawl_time"]    # 爬取时间
    )
    for item in hot_search_data
]


# 使用 MySqlHelper 连接 MySQL
with MySqlHelper() as db:

    # execute_many 用于批量插入多条数据
    # 这里会把 params_list 中的所有热搜数据写入 baidu_hot_search 表
    db.execute_many(insert_sql, params_list)


# 打印插入成功的提示
# len(params_list) 表示本次准备插入的数据条数
print(f"成功存入 {len(params_list)} 条百度热搜数据")

成功存入 10 条百度热搜数据


In [14]:
# 再次使用 MySqlHelper 连接 MySQL
with MySqlHelper() as db:

    # 查询 baidu_hot_search 表中最新插入的 10 条热搜数据
    # ORDER BY crawl_time DESC 表示按爬取时间从新到旧排序
    # rank_no ASC 表示同一次爬取的数据按排名从小到大排序
    rows = db.query_all("""
        SELECT rank_no, keyword, hot_value, url, crawl_time
        FROM baidu_hot_search
        ORDER BY crawl_time DESC, rank_no ASC
        LIMIT 10
    """)


# 打印查询结果，确认数据已经成功进入 MySQL
for row in rows:
    print(row)

{'rank_no': 1, 'keyword': '习近平同金正恩举行会谈', 'hot_value': '7904592', 'url': 'https://www.baidu.com/s?wd=%E4%B9%A0%E8%BF%91%E5%B9%B3%E5%90%8C%E9%87%91%E6%AD%A3%E6%81%A9%E4%B8%BE%E8%A1%8C%E4%BC%9A%E8%B0%88&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34)}
{'rank_no': 2, 'keyword': '习近平和彭丽媛观看朝鲜文艺演出', 'hot_value': '7808599', 'url': 'https://www.baidu.com/s?wd=%E4%B9%A0%E8%BF%91%E5%B9%B3%E5%92%8C%E5%BD%AD%E4%B8%BD%E5%AA%9B%E8%A7%82%E7%9C%8B%E6%9C%9D%E9%B2%9C%E6%96%87%E8%89%BA%E6%BC%94%E5%87%BA&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34)}
{'rank_no': 3, 'keyword': '“李华 我再也不用见到你了”', 'hot_value': '7712263', 'url': 'https://www.baidu.com/s?wd=%E2%80%9C%E6%9D%8E%E5%8D%8E+%E6%88%91%E5%86%8D%E4%B9%9F%E4%B8%8D%E7%94%A8%E8%A7%81%E5%88%B0%E4%BD%A0%E4%BA%86%E2%80%9D&sa=fyb_news&rsv_dl=fyb_news', 'crawl_time': datetime.datetime(2026, 6, 8, 15, 25, 34)}
{'rank_no': 4, 'keyword': '中朝传统友谊崭篇章如何续写', 'hot_value': '7615954', 'url': 'ht